In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

checkpoint_path = "/content/drive/MyDrive/greenmlops/models/resnet18_baseline.pt"
print(os.path.exists(checkpoint_path))

Mounted at /content/drive
True


In [2]:
pip install scikit-learn

In [3]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import numpy as np

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")

model = torchvision.models.resnet18(weights=None)
model.fc = nn.Linear(512, 100)
model.load_state_dict(
    torch.load(checkpoint_path, map_location=DEVICE)
)
model = model.to(DEVICE)
model.eval()

embeddings_buffer = []

def avgpool_hook(module, input, output):
    embeddings_buffer.append(output.detach().cpu().numpy())

model.avgpool.register_forward_hook(avgpool_hook)
print("model loaded, hook attached")

device: cuda
model loaded, hook attached


In [4]:
from torch.utils.data import DataLoader, Subset

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5071, 0.4865, 0.4409],
        std=[0.2673, 0.2564, 0.2762]
    )
])

trainset = torchvision.datasets.CIFAR100(
    root="/content/cifar100", train=True, download=True, transform=transform
)

def extract_embeddings(dataset, indices):
    embeddings_buffer.clear()
    loader = DataLoader(
        Subset(dataset, indices),
        batch_size=64,
        shuffle=False,
        num_workers=2
    )
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(DEVICE)
            _ = model(images)
    raw = np.concatenate(embeddings_buffer, axis=0)
    return raw.reshape(raw.shape[0], -1)

print("dataset loaded, extractor ready")
print(f"total training samples: {len(trainset)}")

100%|██████████| 169M/169M [00:10<00:00, 15.4MB/s]


dataset loaded, extractor ready
total training samples: 50000


In [5]:
SAMPLES_PER_DAY = 85
REF_DAYS = 7
ref_size = SAMPLES_PER_DAY * REF_DAYS

ref_embeddings = extract_embeddings(trainset, list(range(ref_size)))
print(f"reference embeddings shape: {ref_embeddings.shape}")
print(f"expected: ({ref_size}, 512)")

reference embeddings shape: (595, 512)
expected: (595, 512)


In [6]:
SIM_DAYS = 60

day_embeddings = []

for day in range(SIM_DAYS):
    start = (ref_size + day * SAMPLES_PER_DAY) % len(trainset)
    end = start + SAMPLES_PER_DAY

    if end <= len(trainset):
        indices = list(range(start, end))
    else:
        indices = list(range(start, len(trainset))) + list(range(0, end - len(trainset)))

    emb = extract_embeddings(trainset, indices)
    day_embeddings.append(emb)

    if (day + 1) % 10 == 0:
        print(f"day {day+1:02d}/60 done — shape: {emb.shape}")

print("all 60 days extracted")

day 10/60 done — shape: (85, 512)
day 20/60 done — shape: (85, 512)
day 30/60 done — shape: (85, 512)
day 40/60 done — shape: (85, 512)
day 50/60 done — shape: (85, 512)
day 60/60 done — shape: (85, 512)
all 60 days extracted


In [7]:
from sklearn.decomposition import PCA
import pickle

pca = PCA(n_components=50, random_state=42)
ref_pca = pca.fit_transform(ref_embeddings)

explained = pca.explained_variance_ratio_.sum()
print(f"PCA-50 explained variance: {explained:.4f}")
print(f"ref_pca shape: {ref_pca.shape}")

PCA-50 explained variance: 0.8416
ref_pca shape: (595, 50)


In [8]:
def rbf_kernel_mmd(X, Y, bandwidth=None):
    XY = np.vstack([X, Y])
    if bandwidth is None:
        dists = np.sum((XY[:, None] - XY[None, :]) ** 2, axis=-1)
        bandwidth = np.median(dists[dists > 0])

    def gram(A, B):
        diff = A[:, None] - B[None, :]
        return np.exp(-np.sum(diff ** 2, axis=-1) / (2 * bandwidth))

    K_XX = gram(X, X)
    K_YY = gram(Y, Y)
    K_XY = gram(X, Y)

    n, m = len(X), len(Y)
    mmd = (
        (K_XX.sum() - np.trace(K_XX)) / (n * (n - 1))
        + (K_YY.sum() - np.trace(K_YY)) / (m * (m - 1))
        - 2 * K_XY.mean()
    )
    return float(mmd)

N_PAIRS = 500
SAMPLE_SIZE = 256
rng = np.random.default_rng(42)

bandwidth = None
first_pair_X = ref_pca[rng.choice(len(ref_pca), SAMPLE_SIZE, replace=True)]
first_pair_Y = ref_pca[rng.choice(len(ref_pca), SAMPLE_SIZE, replace=True)]
dists = np.sum((np.vstack([first_pair_X, first_pair_Y])[:, None]
                - np.vstack([first_pair_X, first_pair_Y])[None, :]) ** 2, axis=-1)
bandwidth = float(np.median(dists[dists > 0]))
print(f"bandwidth (median heuristic): {bandwidth:.6f}")

null_mmd_values = []
for i in range(N_PAIRS):
    idx_x = rng.choice(len(ref_pca), SAMPLE_SIZE, replace=True)
    idx_y = rng.choice(len(ref_pca), SAMPLE_SIZE, replace=True)
    mmd_val = rbf_kernel_mmd(ref_pca[idx_x], ref_pca[idx_y], bandwidth=bandwidth)
    null_mmd_values.append(mmd_val)
    if (i + 1) % 100 == 0:
        print(f"null pairs computed: {i+1}/500")

null_mmd_values = np.array(null_mmd_values)
null_mean = float(null_mmd_values.mean())
null_std = float(null_mmd_values.std())
threshold = null_mean + 2 * null_std

print(f"\nnull MMD mean:  {null_mean:.6f}")
print(f"null MMD std:   {null_std:.6f}")
print(f"threshold:      {threshold:.6f}")

bandwidth (median heuristic): 1303.320679
null pairs computed: 100/500
null pairs computed: 200/500
null pairs computed: 300/500
null pairs computed: 400/500
null pairs computed: 500/500

null MMD mean:  0.000052
null MMD std:   0.000719
threshold:      0.001489


In [9]:
OUTPUT_DIR = "/content/drive/MyDrive/greenmlops/airflow/include/data/embeddings/cifar100"
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.save(f"{OUTPUT_DIR}/ref_embeddings.npy", ref_embeddings)

for day, emb in enumerate(day_embeddings):
    np.save(f"{OUTPUT_DIR}/day_{day:02d}_embeddings.npy", emb)

with open(f"{OUTPUT_DIR}/pca_model.pkl", "wb") as f:
    pickle.dump(pca, f)

null_stats = np.array([null_mean, null_std, threshold, bandwidth])
np.save(f"{OUTPUT_DIR}/mmd_null_stats.npy", null_stats)

print("saved:")
print(f"  ref_embeddings.npy       {ref_embeddings.shape}")
print(f"  day_00 to day_59 .npy    ({SIM_DAYS} files, each {day_embeddings[0].shape})")
print(f"  pca_model.pkl")
print(f"  mmd_null_stats.npy       [mean, std, threshold, bandwidth]")
print(f"\nall files at: {OUTPUT_DIR}")

saved:
  ref_embeddings.npy       (595, 512)
  day_00 to day_59 .npy    (60 files, each (85, 512))
  pca_model.pkl
  mmd_null_stats.npy       [mean, std, threshold, bandwidth]

all files at: /content/drive/MyDrive/greenmlops/airflow/include/data/embeddings/cifar100


In [10]:
files = sorted(os.listdir(OUTPUT_DIR))
print(f"total files: {len(files)}")
print(f"expected: 63  (1 ref + 60 days + 1 pca + 1 null_stats)")

loaded_stats = np.load(f"{OUTPUT_DIR}/mmd_null_stats.npy")
print(f"\nmmd_null_stats: mean={loaded_stats[0]:.6f}, std={loaded_stats[1]:.6f}, "
      f"threshold={loaded_stats[2]:.6f}, bandwidth={loaded_stats[3]:.6f}")

test_day = np.load(f"{OUTPUT_DIR}/day_00_embeddings.npy")
print(f"day_00 shape: {test_day.shape}  expected: (85, 512)")

total files: 63
expected: 63  (1 ref + 60 days + 1 pca + 1 null_stats)

mmd_null_stats: mean=0.000052, std=0.000719, threshold=0.001489, bandwidth=1303.320679
day_00 shape: (85, 512)  expected: (85, 512)


In [11]:
import os

output_dir = "/content/drive/MyDrive/greenmlops/airflow/include/data/embeddings/cifar100"
files = sorted(os.listdir(output_dir))
npy_days = [f for f in files if f.startswith("day_")]
print(f"day files: {len(npy_days)}  (expected 60)")
print(f"ref_embeddings.npy present: {'ref_embeddings.npy' in files}")
print(f"pca_model.pkl present: {'pca_model.pkl' in files}")
print(f"mmd_null_stats.npy present: {'mmd_null_stats.npy' in files}")

day files: 60  (expected 60)
ref_embeddings.npy present: True
pca_model.pkl present: True
mmd_null_stats.npy present: True


In [12]:
CIFAR100_MEAN = np.array([0.5071, 0.4865, 0.4409])
CIFAR100_STD  = np.array([0.2673, 0.2564, 0.2762])

per_channel_std = CIFAR100_STD
noise_sigma = 0.5 * per_channel_std
print(f"noise_sigma per channel: {noise_sigma}")

noise_sigma per channel: [0.13365 0.1282  0.1381 ]


In [16]:
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import torch

trainset_raw = torchvision.datasets.CIFAR100(
    root="/content/cifar100", train=True, download=False, transform=None
)

class NoisySubset(Dataset):
    def __init__(self, base_dataset, indices, noise_sigma):
        self.base = base_dataset
        self.indices = indices
        self.noise_sigma = torch.tensor(noise_sigma, dtype=torch.float32)

        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.5071, 0.4865, 0.4409],
                std=[0.2673, 0.2564, 0.2762]
            )
        ])

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img, label = self.base[self.indices[idx]]
        tensor = self.transform(img)
        noise = torch.zeros_like(tensor)
        for c in range(3):
            noise[c] = torch.randn_like(tensor[c]) * self.noise_sigma[c]
        return tensor + noise, label

print("NoisySubset defined")

NoisySubset defined


In [17]:
def extract_embeddings_noisy(base_dataset, indices, noise_sigma):
    embeddings_buffer.clear()
    noisy_ds = NoisySubset(base_dataset, indices, noise_sigma)
    loader = torch.utils.data.DataLoader(
        noisy_ds,
        batch_size=64,
        shuffle=False,
        num_workers=2
    )
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(DEVICE)
            _ = model(images)
    raw = np.concatenate(embeddings_buffer, axis=0)
    return raw.reshape(raw.shape[0], -1)

print("noisy extractor defined")

noisy extractor defined


In [18]:
CIFAR100_INJECTION_DAYS = {
    3, 6, 9, 13, 16, 19, 22, 25, 28, 31,
    34, 37, 40, 43, 46, 49, 51, 54, 57, 59
}

for day in CIFAR100_INJECTION_DAYS:
    start = (ref_size + day * SAMPLES_PER_DAY) % len(trainset_raw)
    end = start + SAMPLES_PER_DAY

    if end <= len(trainset_raw):
        indices = list(range(start, end))
    else:
        indices = list(range(start, len(trainset_raw))) + list(range(0, end - len(trainset_raw)))

    emb = extract_embeddings_noisy(trainset_raw, indices, noise_sigma)
    np.save(f"{OUTPUT_DIR}/day_{day:02d}_embeddings_injected.npy", emb)
    print(f"day {day:02d} injected embeddings saved — shape: {emb.shape}")

print("all injection day embeddings saved")

day 03 injected embeddings saved — shape: (85, 512)
day 06 injected embeddings saved — shape: (85, 512)
day 09 injected embeddings saved — shape: (85, 512)
day 13 injected embeddings saved — shape: (85, 512)
day 16 injected embeddings saved — shape: (85, 512)
day 19 injected embeddings saved — shape: (85, 512)
day 22 injected embeddings saved — shape: (85, 512)
day 25 injected embeddings saved — shape: (85, 512)
day 28 injected embeddings saved — shape: (85, 512)
day 31 injected embeddings saved — shape: (85, 512)
day 34 injected embeddings saved — shape: (85, 512)
day 37 injected embeddings saved — shape: (85, 512)
day 40 injected embeddings saved — shape: (85, 512)
day 43 injected embeddings saved — shape: (85, 512)
day 46 injected embeddings saved — shape: (85, 512)
day 49 injected embeddings saved — shape: (85, 512)
day 51 injected embeddings saved — shape: (85, 512)
day 54 injected embeddings saved — shape: (85, 512)
day 57 injected embeddings saved — shape: (85, 512)
day 59 injec

In [19]:
test_injection_day = 3

clean_emb    = np.load(f"{OUTPUT_DIR}/day_{test_injection_day:02d}_embeddings.npy")
injected_emb = np.load(f"{OUTPUT_DIR}/day_{test_injection_day:02d}_embeddings_injected.npy")

clean_pca    = pca.transform(clean_emb)
injected_pca = pca.transform(injected_emb)
ref_pca_check = pca.transform(ref_embeddings)

rng2 = np.random.default_rng(42)

idx_ref  = rng2.choice(len(ref_pca_check), SAMPLE_SIZE, replace=True)
idx_clean = rng2.choice(len(clean_pca), SAMPLE_SIZE, replace=True)
idx_inj   = rng2.choice(len(injected_pca), SAMPLE_SIZE, replace=True)

mmd_clean    = rbf_kernel_mmd(ref_pca_check[idx_ref], clean_pca[idx_clean], bandwidth=bandwidth)
mmd_injected = rbf_kernel_mmd(ref_pca_check[idx_ref], injected_pca[idx_inj], bandwidth=bandwidth)

print(f"MMD clean vs ref:    {mmd_clean:.6f}  (expect < {threshold:.6f})")
print(f"MMD injected vs ref: {mmd_injected:.6f}  (expect > {threshold:.6f})")
print(f"threshold:           {threshold:.6f}")
print(f"injection detectable: {mmd_injected > threshold}")

MMD clean vs ref:    0.005071  (expect < 0.001489)
MMD injected vs ref: 0.008039  (expect > 0.001489)
threshold:           0.001489
injection detectable: True


In [20]:
files = sorted(os.listdir(OUTPUT_DIR))
injected_files = [f for f in files if f.endswith("_injected.npy")]
clean_day_files = [f for f in files if f.startswith("day_") and not f.endswith("_injected.npy")]

print(f"clean day files:    {len(clean_day_files)}  (expected 60)")
print(f"injected day files: {len(injected_files)}   (expected 20)")
print(f"total files:        {len(files)}             (expected 83)")

clean day files:    60  (expected 60)
injected day files: 20   (expected 20)
total files:        83             (expected 83)


In [21]:
rng3 = np.random.default_rng(42)

cross_day_mmd_values = []

for i in range(N_PAIRS):
    day_a = rng3.integers(0, SIM_DAYS)
    day_b = rng3.integers(0, SIM_DAYS)

    emb_a = np.load(f"{OUTPUT_DIR}/day_{day_a:02d}_embeddings.npy")
    emb_b = np.load(f"{OUTPUT_DIR}/day_{day_b:02d}_embeddings.npy")

    pca_a = pca.transform(emb_a)
    pca_b = pca.transform(emb_b)

    idx_a = rng3.choice(len(pca_a), SAMPLE_SIZE, replace=True)
    idx_b = rng3.choice(len(pca_b), SAMPLE_SIZE, replace=True)

    mmd_val = rbf_kernel_mmd(pca_a[idx_a], pca_b[idx_b], bandwidth=bandwidth)
    cross_day_mmd_values.append(mmd_val)

cross_day_mmd_values = np.array(cross_day_mmd_values)
new_null_mean = float(cross_day_mmd_values.mean())
new_null_std  = float(cross_day_mmd_values.std())
new_threshold = new_null_mean + 2 * new_null_std

print(f"cross-day null MMD mean:  {new_null_mean:.6f}")
print(f"cross-day null MMD std:   {new_null_std:.6f}")
print(f"new threshold:            {new_threshold:.6f}")
print(f"clean MMD:                0.005071")
print(f"injected MMD:             0.008039")
print(f"clean below threshold:    {0.005071 < new_threshold}")
print(f"injected above threshold: {0.008039 > new_threshold}")

cross-day null MMD mean:  0.008672
cross-day null MMD std:   0.002916
new threshold:            0.014505
clean MMD:                0.005071
injected MMD:             0.008039
clean below threshold:    True
injected above threshold: False


In [22]:
from scipy import stats

def ks_drift_check(ref_pca, current_pca, threshold_fraction=0.10, n_components=50):
    n_required = int(np.ceil(threshold_fraction * n_components))
    p_values = []
    for i in range(n_components):
        _, p = stats.ks_2samp(ref_pca[:, i], current_pca[:, i])
        p_values.append(p)
    n_significant = sum(p < 0.05 for p in p_values)
    drift_detected = n_significant >= n_required
    return drift_detected, n_significant, p_values

clean_day_emb    = np.load(f"{OUTPUT_DIR}/day_03_embeddings.npy")
injected_day_emb = np.load(f"{OUTPUT_DIR}/day_03_embeddings_injected.npy")

clean_pca_day    = pca.transform(clean_day_emb)
injected_pca_day = pca.transform(injected_day_emb)

drift_clean, n_sig_clean, _ = ks_drift_check(ref_pca, clean_pca_day)
drift_injected, n_sig_injected, _ = ks_drift_check(ref_pca, injected_pca_day)

print(f"clean day 03 — components significant: {n_sig_clean}/50  drift: {drift_clean}  (expect False)")
print(f"injected day 03 — components significant: {n_sig_injected}/50  drift: {drift_injected}  (expect True)")
print(f"required threshold: >= 6 components (>=10% of 50)")

clean day 03 — components significant: 3/50  drift: False  (expect False)
injected day 03 — components significant: 3/50  drift: False  (expect True)
required threshold: >= 6 components (>=10% of 50)


In [23]:
sigma_candidates = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]

rng4 = np.random.default_rng(42)

print(f"{'sigma_multiplier':<18} {'n_sig_clean':<14} {'n_sig_injected':<16} {'clean_ok':<10} {'inject_ok'}")
print("-" * 70)

for sigma_mult in sigma_candidates:
    test_sigma = sigma_mult * per_channel_std

    noisy_ds_test = NoisySubset(trainset_raw, list(range(595, 595 + 85)), test_sigma)
    loader_test = torch.utils.data.DataLoader(
        noisy_ds_test, batch_size=64, shuffle=False, num_workers=2
    )
    embeddings_buffer.clear()
    with torch.no_grad():
        for images, _ in loader_test:
            images = images.to(DEVICE)
            _ = model(images)
    raw = np.concatenate(embeddings_buffer, axis=0)
    test_injected_emb = raw.reshape(raw.shape[0], -1)

    test_injected_pca = pca.transform(test_injected_emb)
    test_clean_pca    = pca.transform(clean_day_emb)

    _, n_clean, _    = ks_drift_check(ref_pca, test_clean_pca)
    _, n_injected, _ = ks_drift_check(ref_pca, test_injected_pca)

    clean_ok  = n_clean < 6
    inject_ok = n_injected >= 6

    print(f"{sigma_mult:<18} {n_clean:<14} {n_injected:<16} {str(clean_ok):<10} {inject_ok}")


sigma_multiplier   n_sig_clean    n_sig_injected   clean_ok   inject_ok
----------------------------------------------------------------------
0.5                3              7                True       True
1.0                3              23               True       True
1.5                3              37               True       True
2.0                3              46               True       True
2.5                3              48               True       True
3.0                3              47               True       True


In [24]:
for day in CIFAR100_INJECTION_DAYS:
    start = (ref_size + day * SAMPLES_PER_DAY) % len(trainset_raw)
    end = start + SAMPLES_PER_DAY

    if end <= len(trainset_raw):
        indices = list(range(start, end))
    else:
        indices = list(range(start, len(trainset_raw))) + list(range(0, end - len(trainset_raw)))

    emb = extract_embeddings_noisy(trainset_raw, indices, noise_sigma)
    np.save(f"{OUTPUT_DIR}/day_{day:02d}_embeddings_injected.npy", emb)
    print(f"day {day:02d} re-extracted — shape: {emb.shape}")

print("all 20 injection day embeddings re-extracted")

day 03 re-extracted — shape: (85, 512)
day 06 re-extracted — shape: (85, 512)
day 09 re-extracted — shape: (85, 512)
day 13 re-extracted — shape: (85, 512)
day 16 re-extracted — shape: (85, 512)
day 19 re-extracted — shape: (85, 512)
day 22 re-extracted — shape: (85, 512)
day 25 re-extracted — shape: (85, 512)
day 28 re-extracted — shape: (85, 512)
day 31 re-extracted — shape: (85, 512)
day 34 re-extracted — shape: (85, 512)
day 37 re-extracted — shape: (85, 512)
day 40 re-extracted — shape: (85, 512)
day 43 re-extracted — shape: (85, 512)
day 46 re-extracted — shape: (85, 512)
day 49 re-extracted — shape: (85, 512)
day 51 re-extracted — shape: (85, 512)
day 54 re-extracted — shape: (85, 512)
day 57 re-extracted — shape: (85, 512)
day 59 re-extracted — shape: (85, 512)
all 20 injection day embeddings re-extracted


In [25]:
clean_day_emb    = np.load(f"{OUTPUT_DIR}/day_03_embeddings.npy")
injected_day_emb = np.load(f"{OUTPUT_DIR}/day_03_embeddings_injected.npy")

clean_pca_day    = pca.transform(clean_day_emb)
injected_pca_day = pca.transform(injected_day_emb)

drift_clean, n_sig_clean, _       = ks_drift_check(ref_pca, clean_pca_day)
drift_injected, n_sig_injected, _ = ks_drift_check(ref_pca, injected_pca_day)

print(f"clean day 03    — components significant: {n_sig_clean}/50    drift: {drift_clean}    (expect False)")
print(f"injected day 03 — components significant: {n_sig_injected}/50    drift: {drift_injected}    (expect True)")

clean day 03    — components significant: 3/50    drift: False    (expect False)
injected day 03 — components significant: 5/50    drift: True    (expect True)


In [26]:
print(f"{'day':<6} {'n_sig_clean':<14} {'n_sig_injected':<16} {'clean_ok':<10} {'inject_ok'}")
print("-" * 55)

for day in sorted(CIFAR100_INJECTION_DAYS):
    clean_emb_d    = np.load(f"{OUTPUT_DIR}/day_{day:02d}_embeddings.npy")
    injected_emb_d = np.load(f"{OUTPUT_DIR}/day_{day:02d}_embeddings_injected.npy")

    clean_pca_d    = pca.transform(clean_emb_d)
    injected_pca_d = pca.transform(injected_emb_d)

    _, n_clean, _    = ks_drift_check(ref_pca, clean_pca_d)
    _, n_injected, _ = ks_drift_check(ref_pca, injected_pca_d)

    print(f"{day:<6} {n_clean:<14} {n_injected:<16} {str(n_clean < 6):<10} {n_injected >= 6}")

day    n_sig_clean    n_sig_injected   clean_ok   inject_ok
-------------------------------------------------------
3      3              5                True       False
6      6              6                False      True
9      2              4                True       False
13     5              8                True       True
16     2              5                True       False
19     1              5                True       False
22     3              8                True       True
25     5              6                True       True
28     0              7                True       True
31     2              6                True       True
34     5              6                True       True
37     2              6                True       True
40     4              7                True       True
43     3              13               True       True
46     0              5                True       False
49     4              9                True       True

In [27]:
noise_sigma_15 = 1.5 * per_channel_std
print(f"updated noise_sigma: {noise_sigma_15}")

for day in CIFAR100_INJECTION_DAYS:
    start = (ref_size + day * SAMPLES_PER_DAY) % len(trainset_raw)
    end = start + SAMPLES_PER_DAY

    if end <= len(trainset_raw):
        indices = list(range(start, end))
    else:
        indices = list(range(start, len(trainset_raw))) + list(range(0, end - len(trainset_raw)))

    emb = extract_embeddings_noisy(trainset_raw, indices, noise_sigma_15)
    np.save(f"{OUTPUT_DIR}/day_{day:02d}_embeddings_injected.npy", emb)
    print(f"day {day:02d} re-extracted — shape: {emb.shape}")

print("all 20 injection days re-extracted at 1.5 sigma")

updated noise_sigma: [0.40095 0.3846  0.4143 ]
day 03 re-extracted — shape: (85, 512)
day 06 re-extracted — shape: (85, 512)
day 09 re-extracted — shape: (85, 512)
day 13 re-extracted — shape: (85, 512)
day 16 re-extracted — shape: (85, 512)
day 19 re-extracted — shape: (85, 512)
day 22 re-extracted — shape: (85, 512)
day 25 re-extracted — shape: (85, 512)
day 28 re-extracted — shape: (85, 512)
day 31 re-extracted — shape: (85, 512)
day 34 re-extracted — shape: (85, 512)
day 37 re-extracted — shape: (85, 512)
day 40 re-extracted — shape: (85, 512)
day 43 re-extracted — shape: (85, 512)
day 46 re-extracted — shape: (85, 512)
day 49 re-extracted — shape: (85, 512)
day 51 re-extracted — shape: (85, 512)
day 54 re-extracted — shape: (85, 512)
day 57 re-extracted — shape: (85, 512)
day 59 re-extracted — shape: (85, 512)
all 20 injection days re-extracted at 1.5 sigma


In [28]:
print(f"{'day':<6} {'n_sig_clean':<14} {'n_sig_injected':<16} {'clean_ok':<10} {'inject_ok'}")
print("-" * 55)

all_clean_ok = True
all_inject_ok = True

for day in sorted(CIFAR100_INJECTION_DAYS):
    clean_emb_d    = np.load(f"{OUTPUT_DIR}/day_{day:02d}_embeddings.npy")
    injected_emb_d = np.load(f"{OUTPUT_DIR}/day_{day:02d}_embeddings_injected.npy")

    clean_pca_d    = pca.transform(clean_emb_d)
    injected_pca_d = pca.transform(injected_emb_d)

    _, n_clean, _    = ks_drift_check(ref_pca, clean_pca_d)
    _, n_injected, _ = ks_drift_check(ref_pca, injected_pca_d)

    c_ok = n_clean < 6
    i_ok = n_injected >= 6

    all_clean_ok  = all_clean_ok and c_ok
    all_inject_ok = all_inject_ok and i_ok

    print(f"{day:<6} {n_clean:<14} {n_injected:<16} {str(c_ok):<10} {i_ok}")

print()
print(f"all clean days below threshold: {all_clean_ok}")
print(f"all injection days detected:    {all_inject_ok}")

day    n_sig_clean    n_sig_injected   clean_ok   inject_ok
-------------------------------------------------------
3      3              38               True       True
6      6              36               False      True
9      2              40               True       True
13     5              37               True       True
16     2              43               True       True
19     1              38               True       True
22     3              38               True       True
25     5              39               True       True
28     0              40               True       True
31     2              34               True       True
34     5              36               True       True
37     2              41               True       True
40     4              41               True       True
43     3              37               True       True
46     0              38               True       True
49     4              35               True       True
51  

In [29]:
def ks_drift_check_v2(ref_pca, current_pca, n_required=7, n_components=50):
    p_values = []
    for i in range(n_components):
        _, p = stats.ks_2samp(ref_pca[:, i], current_pca[:, i])
        p_values.append(p)
    n_significant = sum(p < 0.05 for p in p_values)
    drift_detected = n_significant >= n_required
    return drift_detected, n_significant

print(f"{'day':<6} {'n_sig_clean':<14} {'n_sig_injected':<16} {'clean_ok':<10} {'inject_ok'}")
print("-" * 55)

all_clean_ok  = True
all_inject_ok = True

for day in sorted(CIFAR100_INJECTION_DAYS):
    clean_emb_d    = np.load(f"{OUTPUT_DIR}/day_{day:02d}_embeddings.npy")
    injected_emb_d = np.load(f"{OUTPUT_DIR}/day_{day:02d}_embeddings_injected.npy")

    clean_pca_d    = pca.transform(clean_emb_d)
    injected_pca_d = pca.transform(injected_emb_d)

    _, n_clean    = ks_drift_check_v2(ref_pca, clean_pca_d)
    _, n_injected = ks_drift_check_v2(ref_pca, injected_pca_d)

    c_ok = n_clean < 7
    i_ok = n_injected >= 7

    all_clean_ok  = all_clean_ok and c_ok
    all_inject_ok = all_inject_ok and i_ok

    print(f"{day:<6} {n_clean:<14} {n_injected:<16} {str(c_ok):<10} {i_ok}")

print()
print(f"all clean days below threshold: {all_clean_ok}")
print(f"all injection days detected:    {all_inject_ok}")

day    n_sig_clean    n_sig_injected   clean_ok   inject_ok
-------------------------------------------------------
3      3              38               True       True
6      6              36               True       True
9      2              40               True       True
13     5              37               True       True
16     2              43               True       True
19     1              38               True       True
22     3              38               True       True
25     5              39               True       True
28     0              40               True       True
31     2              34               True       True
34     5              36               True       True
37     2              41               True       True
40     4              41               True       True
43     3              37               True       True
46     0              38               True       True
49     4              35               True       True
51  